# 04_Exploratory Data Analysis (EDA)

## 1. Objectifs de l'analyse exploratoire

Cette étape propose une première lecture des données AdventureWorks après leur préparation et leur validation.

Elle vise à :
- prendre connaissance du volume et de la structure des tables ;
- examiner les principales distributions numériques de la table de ventes ;
- calculer les indicateurs commerciaux globaux ;
- repérer les premières caractéristiques qui guideront les analyses thématiques.

Les analyses détaillées des ventes, des produits et des clients sont développées respectivement dans les notebooks 05, 06 et 07. Les analyses avancées seront regroupées dans le notebook 08, tandis que les conclusions métier seront centralisées dans le notebook 09.

In [2]:
# Importer les bibliothèques nécessaires
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Définir le chemin du projet
BASE_DIR = Path.cwd().parent
PROCESSED_DIR = BASE_DIR / "data" / "processed"

# Charger les cinq tables préparées
dim_date = pd.read_csv(
    PROCESSED_DIR / "DimDate.csv",
    parse_dates=["Date"]
)

dim_customer = pd.read_csv(
    PROCESSED_DIR / "DimCustomer.csv",
    parse_dates=["BirthDate", "DateFirstPurchase"]
)

dim_product = pd.read_csv(
    PROCESSED_DIR / "DimProduct.csv",
    parse_dates=["StartDate"]
)

fact_sales = pd.read_csv(
    PROCESSED_DIR / "FactInternetSales.csv",
    parse_dates=["OrderDate", "ShipDate"]
)

dim_territory = pd.read_csv(
    PROCESSED_DIR / "DimSalesTerritory.csv"
)

print("Chargement des données préparées terminé.")

Chargement des données préparées terminé.


In [3]:
# Regrouper les tables dans un dictionnaire
tables = {
    "DimDate": dim_date,
    "DimCustomer": dim_customer,
    "DimProduct": dim_product,
    "FactInternetSales": fact_sales,
    "DimSalesTerritory": dim_territory
}

# Afficher les dimensions des tables
for nom, table in tables.items():
    print(
        f"{nom} : {table.shape[0]:,} lignes "
        f"| {table.shape[1]} colonnes"
    )

DimDate : 1,461 lignes | 16 colonnes
DimCustomer : 18,484 lignes | 17 colonnes
DimProduct : 606 lignes | 13 colonnes
FactInternetSales : 58,189 lignes | 15 colonnes
DimSalesTerritory : 11 lignes | 5 colonnes


In [4]:
# Sélectionner les principales variables numériques
colonnes_numeriques = [
    "OrderQuantity",
    "UnitPrice",
    "TotalProductCost",
    "CalculatedTotalCost",
    "SalesAmount",
    "TaxAmt"
]

# Afficher les statistiques descriptives
statistiques = fact_sales[colonnes_numeriques].describe().T

statistiques

,count,mean,std,min,25%,50%,75%,max
OrderQuantity,58189.0,1.569386,1.047532,1.0000,1.0000,1.0000,2.0000,4.0000
UnitPrice,58189.0,413.888218,833.052938,0.5725,4.9900,24.4900,269.9950,3578.2700
TotalProductCost,58189.0,296.539185,560.171436,0.8565,3.3623,12.1924,343.6496,2171.2942
CalculatedTotalCost,58189.0,465.268017,1099.307825,0.8565,3.7363,13.0863,343.6496,8685.1768
SalesAmount,58189.0,503.666270,941.462817,2.2900,8.9900,32.6000,539.9900,3578.2700
TaxAmt,58189.0,40.293303,75.317027,0.1832,0.7192,2.6080,43.1992,286.2616


## 2. Indicateurs commerciaux globaux

Les indicateurs suivants donnent une vue d'ensemble de l'activité couverte par la table `FactInternetSales`.

In [13]:
# Calculer les principaux indicateurs commerciaux
chiffre_affaires = fact_sales["SalesAmount"].sum()
cout_total_estime = fact_sales["CalculatedTotalCost"].sum()
benefice_brut_estime = chiffre_affaires - cout_total_estime

# Le bénéfice rapporté au chiffre d'affaires correspond au taux de marque
taux_marque_estime = (
    benefice_brut_estime / chiffre_affaires * 100
)

nombre_lignes = len(fact_sales)
nombre_commandes = fact_sales["SalesOrderNumber"].nunique()
nombre_clients = fact_sales["CustomerKey"].nunique()
nombre_produits = fact_sales["ProductKey"].nunique()
quantite_totale = fact_sales["OrderQuantity"].sum()

# Calculer le panier moyen global
panier_moyen = chiffre_affaires / nombre_commandes

# Construire un tableau de synthèse
indicateurs_globaux = pd.DataFrame({
    "Indicateur": [
        "Nombre de lignes de vente",
        "Nombre de commandes",
        "Nombre de clients acheteurs",
        "Nombre de produits vendus",
        "Quantité totale vendue",
        "Chiffre d'affaires",
        "Panier moyen",
        "Coût total estimé",
        "Bénéfice brut estimé",
        "Taux de marque estimé"
        
    ],
    "Valeur": [
        nombre_lignes,
        nombre_commandes,
        nombre_clients,
        nombre_produits,
        quantite_totale,
        chiffre_affaires,
        panier_moyen,
        cout_total_estime,
        benefice_brut_estime,
        taux_marque_estime
    ]
})

display(indicateurs_globaux.round(2))

,Indicateur,Valeur
0,Nombre de lignes de vente,58189.00
1,Nombre de commandes,26683.00
2,Nombre de clients acheteurs,17918.00
3,Nombre de produits vendus,158.00
4,Quantité totale vendue,91321.00
5,Chiffre d'affaires,29307836.59
6,Panier moyen,1098.37
7,Coût total estimé,27073480.62
8,Bénéfice brut estimé,2234355.98
9,Taux de marque estimé,7.62


## 3. Précaution d'interprétation des indicateurs financiers

La colonne `TotalProductCost` présente dans le fichier préparé correspond à une valeur unitaire du coût standard dans les observations étudiées. Pour estimer le coût total de chaque ligne de vente, le notebook de préparation a créé `CalculatedTotalCost`, calculé en multipliant la quantité vendue par le coût standard unitaire.

Les coûts, bénéfices bruts et taux de marque calculés à partir de cette colonne sont donc des estimations. Ils ne doivent pas être interprétés comme des résultats comptables réels.

## 4. Orientation vers les analyses thématiques

Les premiers indicateurs globaux servent de point de départ. Les analyses approfondies sont réparties dans les notebooks suivants :

- `05_Sales_Performance.ipynb` : évolution des ventes, commandes, rentabilité et géographie ;
- `06_Product_Analytics.ipynb` : catégories, sous-catégories, prix et rentabilité des produits ;
- `07_Customer_Analytics.ipynb` : profils et comportements des clients ;
- `08_Advanced_Analytics.ipynb` : analyses multivariées et approfondissements statistiques ;
- `09_Business_Insights.ipynb` : synthèse transversale et recommandations métier.